In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt')

[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

Étapes pour implémenter un système de recommandation de films de haut niveau

Chargement et préparation des données

Filtrage collaboratif basé sur les utilisateurs

Filtrage basé sur le contenu (à l'aide des caractéristiques des films)

Combinaison des deux approches dans un système hybride

Création d'une interface utilisateur (simple via Streamlit ou Flask)


1. Chargement et Préparation des Données



Nous commençons par charger les données et effectuer un pré-traitement pour gérer les valeurs manquantes et préparer les données pour l'analyse.

In [3]:
data=pd.read_csv("Amazon- Movies and Films.csv")
data.head()

,Unnamed: 0,title,Movie_Rating,No_of_Ratings,Format,ReleaseYear,MPAA_Rating,Directed_By,Starring,Price
0,0,Totally Killer,4.3,323,Prime Video,2023.0,R,Nahnatchka Khan,"Kiernan Shipka,Olivia Holt,Julie Bowen",NaN
1,1,Guy Ritchie's The Covenant,4.7,13268,Prime Video,2023.0,R,Guy Ritchie,"Jake Gyllenhaal,Dar Salim,Antony Starr",5.99
2,2,A Million Miles Away,4.9,1126,Prime Video,2023.0,PG,Alejandra Márquez Abella,"Michael Peña,Rosa Salazar",NaN
3,3,Kelce,5.0,570,Prime Video,2023.0,NaN,Don Argott,"Jason Kelce,Travis Kelce,Kylie Kelce,Connor Ba...",NaN
4,4,Despicable Me 3,4.8,31813,Prime Video,2017.0,PG,"Pierre Coffin,Kyle Balda","Steve Carell,Kristen Wiig,Trey Parker",NaN


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2108 entries, 0 to 2107
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     2108 non-null   int64  
 1   title          2108 non-null   object 
 2   Movie_Rating   2108 non-null   float64
 3   No_of_Ratings  2108 non-null   int64  
 4   Format         2108 non-null   object 
 5   ReleaseYear    2069 non-null   float64
 6   MPAA_Rating    1378 non-null   object 
 7   Directed_By    2043 non-null   object 
 8   Starring       2107 non-null   object 
 9   Price          1011 non-null   float64
dtypes: float64(3), int64(2), object(5)
memory usage: 164.8+ KB


In [5]:

# Nettoyer les données
data_cleaned = data.dropna(subset=['Movie_Rating', 'title', 'Starring'])  # Supprimer les lignes avec des valeurs manquantes
data_cleaned['ReleaseYear'] = data_cleaned['ReleaseYear'].fillna(data_cleaned['ReleaseYear'].mean())  # Remplacer les valeurs manquantes par la moyenne
data_cleaned['Price'] = data_cleaned['Price'].fillna(data_cleaned['Price'].mean())  # Remplacer les valeurs manquantes par la moyenne

# Afficher un échantillon des données nettoyées
print(data_cleaned.head())


   Unnamed: 0                       title  Movie_Rating  No_of_Ratings  \
0           0              Totally Killer           4.3            323   
1           1  Guy Ritchie's The Covenant           4.7          13268   
2           2        A Million Miles Away           4.9           1126   
3           3                       Kelce           5.0            570   
4           4             Despicable Me 3           4.8          31813   

        Format  ReleaseYear MPAA_Rating               Directed_By  \
0  Prime Video       2023.0           R           Nahnatchka Khan   
1  Prime Video       2023.0           R               Guy Ritchie   
2  Prime Video       2023.0          PG  Alejandra Márquez Abella   
3  Prime Video       2023.0         NaN                Don Argott   
4  Prime Video       2017.0          PG  Pierre Coffin,Kyle Balda   

                                            Starring     Price  
0             Kiernan Shipka,Olivia Holt,Julie Bowen  4.830465  
1         

C:\Users\NGOUYE GNING\AppData\Local\Temp\ipykernel_14020\1677915847.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['ReleaseYear'] = data_cleaned['ReleaseYear'].fillna(data_cleaned['ReleaseYear'].mean())  # Remplacer les valeurs manquantes par la moyenne
C:\Users\NGOUYE GNING\AppData\Local\Temp\ipykernel_14020\1677915847.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['Price'] = data_cleaned['Price'].fillna(data_cleaned['Price'].mean())  # Remplacer les valeurs manquantes p

2. Filtrage Collaboratif Basé sur les Utilisateurs


Le filtrage collaboratif repose sur la similarité entre les utilisateurs, en utilisant leurs évaluations pour recommander des films similaires. Nous allons construire une matrice d’évaluations utilisateurs-film.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# Créer une matrice utilisateur-film en utilisant le nombre de notes et les évaluations
movie_ratings_matrix = data_cleaned.pivot_table(index='Starring', columns='title', values='Movie_Rating')

# Appliquer une normalisation pour éviter les biais d'évaluation
scaler = StandardScaler()
movie_ratings_matrix_scaled = scaler.fit_transform(movie_ratings_matrix.fillna(0))

# Calculer la similarité entre les films
cosine_sim = cosine_similarity(movie_ratings_matrix_scaled)

# Fonction pour obtenir les films similaires
def get_similar_movies(movie_title, num_recommendations=5):
    idx = data_cleaned[data_cleaned['title'] == movie_title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    recommended_movies = []
    for i in sim_scores[1:num_recommendations+1]:  # Ignorer le film lui-même
        recommended_movies.append(data_cleaned.iloc[i[0]]['title'])
    
    return recommended_movies

# Exemple d'utilisation
recommended_movies = get_similar_movies("Totally Killer")
print(f"Films similaires à 'Totally Killer':")
for movie in recommended_movies:
    print(movie)


Films similaires à 'Totally Killer':
Crooklyn
Joe Bell
Tim Burton's Corpse Bride
Savannah
Anne & Mary Boleyn - A Tale of Two Sisters


3. Filtrage Basé sur le Contenu



Le filtrage basé sur le contenu recommande des films en fonction des caractéristiques des films eux-mêmes, telles que les acteurs (Starring), le genre (MPAA_Rating), ou l'année de sortie (ReleaseYear).

Étapes :
Utiliser les caractéristiques des films comme les acteurs, le genre, et l'année pour créer un vecteur de caractéristiques.
Calculer la similarité entre les films en utilisant des mesures comme la similarité cosinus.
Code pour filtrage basé sur le contenu :



In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Créer un vecteur de caractéristiques basé sur les acteurs et les genres
movie_features = data_cleaned['Starring'] + " " + data_cleaned['MPAA_Rating'].fillna('')  # Combine les acteurs et les genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movie_features)

# Calculer la similarité entre les films
cosine_sim_content = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Fonction pour obtenir les films similaires basés sur le contenu
def get_content_based_recommendations(movie_title, num_recommendations=5):
    idx = data_cleaned[data_cleaned['title'] == movie_title].index[0]
    sim_scores = list(enumerate(cosine_sim_content[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    recommended_movies = []
    for i in sim_scores[1:num_recommendations+1]:  # Ignorer le film lui-même
        recommended_movies.append(data_cleaned.iloc[i[0]]['title'])
    
    return recommended_movies

# Exemple d'utilisation
recommended_movies_content = get_content_based_recommendations("Totally Killer")
print(f"Films similaires basés sur le contenu à 'Totally Killer':")
for movie in recommended_movies_content:
    print(movie)


Films similaires basés sur le contenu à 'Totally Killer':
Status Update
Season Of Fear
Season Of Fear
One Special Night
Pumpkin Pie Wars


4. Système Hybride (Hybrid)


Un système hybride combine les deux approches : collaboratif et basé sur le contenu. Cela peut améliorer les recommandations en prenant en compte les similitudes basées à la fois sur les utilisateurs et les caractéristiques des films.

Code pour combiner les deux systèmes :

In [8]:
def hybrid_recommendation(movie_title, starring, num_recommendations=5):
    # Filtrage collaboratif
    collaborative_recs = get_similar_movies(movie_title, num_recommendations)
    
    # Filtrage basé sur le contenu
    content_based_recs = get_content_based_recommendations(movie_title, num_recommendations)
    
    # Combiner les deux listes (en évitant les doublons)
    hybrid_recs = list(set(collaborative_recs).union(content_based_recs))
    
    return hybrid_recs

# Exemple d'utilisation
hybrid_recs = hybrid_recommendation("Totally Killer", "Kiernan Shipka,Olivia Holt", num_recommendations=5)
print(f"Films recommandés (hybrides) pour 'Totally Killer':")
for movie in hybrid_recs:
    print(movie)


Films recommandés (hybrides) pour 'Totally Killer':
Season Of Fear
Crooklyn
Anne & Mary Boleyn - A Tale of Two Sisters
One Special Night
Pumpkin Pie Wars
Savannah
Joe Bell
Tim Burton's Corpse Bride
Status Update


5. Interface Utilisateur Simple avec Streamlit


Une interface simple peut être créée avec Streamlit pour rendre le système plus interactif.

In [9]:
import streamlit as st

# Interface Streamlit
st.title('Système de Recommandation de Films')

movie_title = st.text_input('Entrez le titre d\'un film:', 'Totally Killer')

if movie_title:
    recommendations = hybrid_recommendation(movie_title, "", num_recommendations=5)
    st.write(f"Films recommandés similaires à '{movie_title}':")
    for rec in recommendations:
        st.write(f"- {rec}")


2025-01-07 00:27:09.831 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.096 
  command:

    streamlit run c:\ProjectML\gning\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-01-07 00:27:11.097 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.099 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.100 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-07 00:27:11.101 Session state does not functi

6. Évaluation du Système de Recommandation


Enfin, vous pouvez évaluer la performance du système en mesurant des indicateurs comme la précision, le rappel, la F-mesure, ou en calculant l'erreur quadratique moyenne (RMSE).

Code pour calculer l'erreur quadratique moyenne (RMSE) :

In [10]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Exemple pour calculer RMSE entre les évaluations réelles et les évaluations prédites
# Assumons que nous avons un vecteur de prédictions et de valeurs réelles
y_true = [4.5, 3.0, 5.0, 2.5, 4.0]
y_pred = [4.0, 3.2, 4.8, 2.7, 4.1]

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse}")


RMSE: 0.2756809750418045


onclusion
Avec ces étapes, vous avez un système de recommandation de films complet, en utilisant les trois principales approches :

Filtrage collaboratif,
Filtrage basé sur le contenu, et
Système hybride.
Vous pouvez personnaliser et étendre ce système selon les besoins de votre projet, comme en ajoutant plus de données ou en ajustant les hyperparamètres des algorithmes. L'interface utilisateur Streamlit offre une façon simple d'interagir avec le système et de visualiser les recommandations.